# Clase 05 — Clasificación de textos

**Curso:** NLP Clásico con Python · **Bloque 2:** Representar y clasificar
**Tiempo estimado:** 110 minutos · **Capítulo del manual:** `manual/05_clasificacion_de_textos.md`

## Qué vas a ver en esta clase

- **Naive Bayes multinomial** calculado **a mano** sobre un ejemplo minúsculo, con el suavizado de Laplace, y comprobado con `MultinomialNB`.
- **Regresión logística** y **SVM lineal** (`LinearSVC`) sobre TF-IDF, comparadas con Naive Bayes en tickets en español y en italiano.
- La **`Pipeline`** y la **fuga de datos**: un experimento real, con el vectorizador fuera y dentro de la `Pipeline`.
- **Métricas:** precisión, *recall*, F1 (macro y ponderado) y matriz de confusión, calculadas a mano y con `classification_report`.
- **Clases desbalanceadas:** creamos el desbalance submuestreando y vemos por qué la exactitud engaña.
- **Leer los coeficientes** del modelo, con sentido crítico: los tickets tienen artefactos que se cuelan entre las palabras más pesadas.
- **Validación cruzada estratificada** (`StratifiedKFold`).

## Requisitos

- Bolsa de palabras y TF-IDF con `TfidfVectorizer` (clase 4) y por qué `fit` se hace solo con el entrenamiento.
- Python con `numpy` y `pandas`. Librerías: `scikit-learn`, `matplotlib`. La celda siguiente instala solo lo que falte (funciona igual en Google Colab).
- La carpeta `datos/` del curso (`tickets_es.csv` y `tickets_it.csv`). En Colab, súbela desde el panel de archivos.

> **Cómo usar este notebook:** ejecuta las celdas en orden. Los cálculos a mano se comprueban con scikit-learn mediante un `assert`. **Aviso:** los tickets en español y en italiano son adaptaciones muy cercanas entre sí, así que **no son dos experimentos independientes**; y son corpus pequeños y sintéticos escritos para el curso (*ejemplo ilustrativo*): las cifras no se generalizan a textos reales.

---
## Instalación

Instala solo lo que falte. En un entorno con todo instalado no hace nada.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("scipy", "scipy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, fijamos la **semilla**, definimos `cargar_datos` (busca `datos/`, `../datos/` y `/content/datos/`) y leemos los 400 tickets de cada idioma (100 por categoría: `cuenta`, `devoluciones`, `envio`, `facturacion`). Dividimos **una sola vez** en 75 % entrenamiento y 25 % prueba, **estratificado** por categoría, para que la prueba tenga 25 tickets de cada una. También definimos `crear_pipeline`, que usaremos toda la clase.

In [ ]:
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

random.seed(42)                          # semilla fija: todos obtenemos los mismos resultados
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = False
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

ARCHIVO_TICKETS = {"spa": "tickets_es.csv", "ita": "tickets_it.csv"}
tickets = {idioma: pd.read_csv(cargar_datos(archivo)) for idioma, archivo in ARCHIVO_TICKETS.items()}
ETIQUETAS = sorted(tickets["spa"]["categoria"].unique())      # orden alfabético, el mismo que usa scikit-learn


def dividir(df):
    """75 % entrenamiento / 25 % prueba, estratificado por categoría."""
    return train_test_split(df["texto"], df["categoria"], test_size=0.25, stratify=df["categoria"], random_state=42)


DIV = {idioma: dividir(df) for idioma, df in tickets.items()}


def crear_pipeline(clasificador, **opciones):
    """TF-IDF (unigramas y bigramas) + clasificador, en una sola Pipeline."""
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), **opciones)), ("clf", clasificador)])


def separar_veredicto(texto):
    """Separa la última frase si tiene 4 palabras o menos (veredicto o etiqueta de asunto)."""
    m = re.search(r"(?<=[.?!])\s+([^.?!]+[.?!])\s*$", texto)
    if m and len(m.group(1).split()) <= 4:
        return texto[:m.start()].strip(), m.group(1).strip()
    return texto, ""


print({idioma: (len(d[0]), len(d[1])) for idioma, d in DIV.items()}, "(entrenamiento, prueba)")
print("Datos y librerías cargados correctamente.")

---
## 1. Punto de partida: qué es un buen resultado

Un clasificador supervisado aprende de ejemplos con etiqueta (`texto` → `categoria`) y se evalúa en ejemplos que **no ha visto**. Antes de entrenar nada, conviene fijar una **línea base**: el resultado de un modelo trivial. Si un modelo no la supera con claridad, no ha aprendido nada. Aquí, un `DummyClassifier` que predice siempre la misma clase.

In [ ]:
for idioma in ("spa", "ita"):
    X_ent, X_prueba, y_ent, y_prueba = DIV[idioma]
    trivial = DummyClassifier(strategy="most_frequent").fit(X_ent, y_ent)
    print(idioma, "entrenamiento:", dict(y_ent.value_counts().sort_index()),
          "| prueba:", dict(y_prueba.value_counts().sort_index()))
    print("   exactitud del modelo trivial:", round(trivial.score(X_prueba, y_prueba), 3))
    assert y_prueba.value_counts().nunique() == 1          # 25 tickets por categoría en la prueba

**¿Qué observas?** Con cuatro categorías equilibradas, adivinar siempre la misma da 0,25: esa es la línea base. Todo lo que hagamos debe superar ese 25 % con holgura. En la prueba hay 100 tickets: **cada ticket vale un punto porcentual**, así que diferencias de uno o dos puntos entre modelos son ruido.

---
## 2. Naive Bayes multinomial a mano, con suavizado de Laplace

Naive Bayes elige la clase con mayor `P(clase) · Π P(palabra | clase)`, suponiendo que las palabras son independientes dentro de cada clase («ingenuo»). Cada `P(palabra | clase)` sale de contar palabras en los textos de esa clase. Problema: una palabra **nunca vista** en una clase tiene probabilidad 0 y anula todo el producto. El **suavizado de Laplace** suma `alpha = 1` a cada recuento:

`P(palabra | clase) = (recuento + alpha) / (N_clase + alpha · V)`

con `N_clase` el total de palabras de la clase y `V` el tamaño del vocabulario. Lo calculamos con un corpus **minúsculo** (dos tickets por clase) y lo comprobamos con `MultinomialNB(alpha=1)`.

In [ ]:
# datos-es
MINI = {
    "spa": {"facturacion": ["cobro doble tarjeta", "factura error cobro"],
            "envio": ["paquete perdido no llega", "no llega el envío"]},
    "ita": {"facturacion": ["addebito doppio carta", "fattura errore addebito"],
            "envio": ["pacco perso non arriva", "non arriva la spedizione"]},
}
PRUEBA = {"spa": "cobro no llega", "ita": "addebito non arriva"}

In [ ]:
def nb_a_mano(mini, texto, alpha=1.0):
    """Naive Bayes multinomial con suavizado: devuelve prioris, tabla de probabilidades y posteriores."""
    docs = [(clase, re.findall(r"\w+", d.lower())) for clase, ds in mini.items() for d in ds]
    vocab = sorted({t for _, toks in docs for t in toks})
    V = len(vocab)
    palabras = re.findall(r"\w+", texto.lower())
    clases = sorted(mini)
    resultado = {}
    for clase in clases:
        toks = [t for c, ts in docs if c == clase for t in ts]
        N = len(toks)
        cuenta = Counter(toks)
        priori = sum(1 for c, _ in docs if c == clase) / len(docs)
        probs = {p: (cuenta[p] + alpha) / (N + alpha * V) for p in palabras}
        verosimilitud = priori * np.prod(list(probs.values()))
        resultado[clase] = {"N": N, "V": V, "priori": priori, "probs": probs, "recuentos": {p: cuenta[p] for p in palabras}, "puntuacion": verosimilitud}
    total = sum(r["puntuacion"] for r in resultado.values())
    for r in resultado.values():
        r["posterior"] = r["puntuacion"] / total if total > 0 else float("nan")      # 0/0 sin suavizado
    return resultado


for idioma in ("spa", "ita"):
    res = nb_a_mano(MINI[idioma], PRUEBA[idioma])
    print("Ticket nuevo:", repr(PRUEBA[idioma]))
    for clase, r in res.items():
        detalle = " · ".join(f"P({p}|{clase[:3]}) = {r['recuentos'][p] + 1}/{r['N'] + r['V']}" for p in r["probs"])
        print(f"  {clase:12s} N = {r['N']}, V = {r['V']}, priori = {r['priori']:.2f} | {detalle}")
        print(f"               puntuación = {r['puntuacion']:.6f}  →  posterior = {r['posterior']:.4f}")

    # Sin suavizado (alpha = 0): los productos se anulan
    sin = nb_a_mano(MINI[idioma], PRUEBA[idioma], alpha=0.0)
    print("  Sin suavizado (alpha = 0), puntuaciones:", {k: v["puntuacion"] for k, v in sin.items()})

    # Comprobación con scikit-learn
    textos = [d for ds in MINI[idioma].values() for d in ds]
    etiquetas = [c for c, ds in MINI[idioma].items() for _ in ds]
    cv = CountVectorizer(token_pattern=r"\w+")
    nb = MultinomialNB(alpha=1.0).fit(cv.fit_transform(textos), etiquetas)
    proba = nb.predict_proba(cv.transform([PRUEBA[idioma]]))[0]
    for clase, p in zip(nb.classes_, proba):
        assert np.isclose(p, res[clase]["posterior"])
    assert res["envio"]["N"] == 8 and res["facturacion"]["N"] == 6 and res["envio"]["V"] == 11
    print("  coincide con MultinomialNB(alpha=1) ✔, predicción:", nb.predict(cv.transform([PRUEBA[idioma]]))[0])

**¿Qué observas?** En español, `facturacion` tiene 6 palabras y `envio` 8; el vocabulario tiene 11. `cobro` aparece 2 veces en `facturacion`: `(2 + 1)/(6 + 11) = 3/17`. Las otras dos palabras del ticket nuevo no aparecen en esa clase y valen `1/17` cada una (**el suavizado evita el cero**). Los posteriores son `facturacion` 0,3176 y `envio` 0,6824: gana `envio`, porque `no llega` pesa más que `cobro`. Con `alpha = 0` **ambas puntuaciones son 0** y no se puede decidir. Un detalle: con textos largos los productos se hacen minúsculos; en la práctica se suman **logaritmos** (scikit-learn lo hace así internamente).

---
## 3. Tres clasificadores sobre TF-IDF

Los tres comparten la misma representación (`TfidfVectorizer` con unigramas y bigramas) y cambian solo el clasificador:

- **`MultinomialNB`**: probabilístico, muy rápido; definido sobre recuentos, pero con TF-IDF funciona bien en la práctica.
- **`LogisticRegression`**: aprende un peso por palabra y clase; da probabilidades y coeficientes que se pueden leer.
- **`LinearSVC`** (SVM lineal): busca el hiperplano que separa las clases con mayor margen; suele ser muy competitivo con texto.

Los metemos en una `Pipeline` para que el vectorizador y el clasificador se ajusten juntos y solo con el entrenamiento.

In [ ]:
MODELOS = {"MultinomialNB": lambda: MultinomialNB(),
           "LogisticRegression": lambda: LogisticRegression(max_iter=1000),
           "LinearSVC": lambda: LinearSVC()}
PREDICCIONES = {}                                  # guardamos las predicciones para usarlas después
filas = []
for nombre, fabrica in MODELOS.items():
    fila = {"modelo": nombre}
    for idioma in ("spa", "ita"):
        X_ent, X_prueba, y_ent, y_prueba = DIV[idioma]
        modelo = crear_pipeline(fabrica()).fit(X_ent, y_ent)
        pred = modelo.predict(X_prueba)
        PREDICCIONES[(nombre, idioma)] = pred
        fila[f"exactitud {idioma}"] = round(accuracy_score(y_prueba, pred), 3)
        fila[f"F1 macro {idioma}"] = round(f1_score(y_prueba, pred, average="macro"), 3)
        assert accuracy_score(y_prueba, pred) > 0.6         # muy por encima del 25 % del modelo trivial
    filas.append(fila)
tabla_modelos = pd.DataFrame(filas)
print(tabla_modelos.to_string(index=False))

**¿Qué observas?** Los tres modelos quedan entre 0,85 y 0,91 de exactitud, muy por encima de 0,25. La diferencia máxima entre modelos, en el mismo idioma, es de 0,04: **unos pocos tickets de 100**. Con una sola partición de prueba no se puede afirmar que uno sea «mejor»; más adelante lo miramos con validación cruzada. Y recuerda que español e italiano son adaptaciones cercanas: que las cifras se parezcan **no** confirma nada por sí mismo.

---
## 4. `Pipeline` y fuga de datos: un experimento real

**Fuga de datos** (*data leakage*): información de la prueba que se cuela en el entrenamiento y hace la evaluación demasiado optimista. El error clásico es ajustar el vectorizador con **todo** el corpus y **después** dividir. La `Pipeline` lo evita: en cada `fit` ajusta el vectorizador solo con los datos que recibe (y dentro de la validación cruzada, solo con el entrenamiento de cada partición).

Vamos a medirlo. Comparamos, con **20 particiones aleatorias** distintas (semillas 0 a 19), la exactitud con el TF-IDF **ajustado con todo el corpus** (`ajuste global`) frente a la `Pipeline` correcta, con 40 y con 300 tickets de entrenamiento. Es un experimento más largo: tarda unos segundos.

In [ ]:
def fuga_vectorizador(df, n_ent, semillas=range(20)):
    """Exactitud media con vectorizador ajustado con todo el corpus frente a Pipeline correcta."""
    global_, correcta = [], []
    for semilla in semillas:
        X_ent, X_prueba, y_ent, y_prueba = train_test_split(df["texto"], df["categoria"], train_size=n_ent, test_size=100,
                                                             stratify=df["categoria"], random_state=semilla)
        vec = TfidfVectorizer(ngram_range=(1, 2)).fit(df["texto"])          # MAL: ve también la prueba
        modelo = LogisticRegression(max_iter=1000).fit(vec.transform(X_ent), y_ent)
        global_.append(modelo.score(vec.transform(X_prueba), y_prueba))
        correcta.append(crear_pipeline(LogisticRegression(max_iter=1000)).fit(X_ent, y_ent).score(X_prueba, y_prueba))
    return np.mean(global_), np.mean(correcta)


filas = []
for idioma in ("spa", "ita"):
    for n_ent in (40, 300):
        g, p = fuga_vectorizador(tickets[idioma], n_ent)
        filas.append({"idioma": idioma, "entrenamiento": n_ent, "ajuste global": round(g, 4), "Pipeline": round(p, 4), "diferencia": round(g - p, 4)})
        assert abs(g - p) < 0.03                       # con TF-IDF la diferencia es pequeña (lo comprobamos, no lo damos por supuesto)
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** Con TF-IDF, **la diferencia es prácticamente nula** (unas milésimas, a veces negativa): el vectorizador solo aprende el vocabulario y los `idf`, **sin mirar las etiquetas**, y con estos corpus el vocabulario global apenas cambia lo que el modelo puede aprender. Es un resultado honesto: no siempre hay fuga visible. Pero eso **no** justifica el atajo: el riesgo real aparece en cuanto un paso de la cadena **usa las etiquetas**. Lo vemos en la celda siguiente.

### Un paso que sí usa las etiquetas: selección de rasgos

Añadimos `SelectKBest(chi2, k=50)`, que se queda con las 50 palabras más asociadas a las etiquetas. Para que el efecto sea inequívoco **barajamos las etiquetas al azar**: ya no hay ninguna relación real entre texto y categoría, y con cuatro clases un modelo honesto no debería pasar del 25 % de aciertos. Comparamos seleccionar con **todo** el corpus y **después** dividir, frente a hacerlo dentro de la `Pipeline`.

In [ ]:
def fuga_seleccion(df, semillas=range(20)):
    """Con etiquetas barajadas: selección de rasgos fuera y dentro de la Pipeline."""
    rng = np.random.RandomState(0)
    y_falsa = pd.Series(rng.permutation(df["categoria"].values), index=df.index)
    X_todo = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(df["texto"])
    X_sel = SelectKBest(chi2, k=50).fit_transform(X_todo, y_falsa)              # MAL: elige las palabras mirando TODAS las etiquetas
    fuera, dentro = [], []
    for semilla in semillas:
        idx_ent, idx_prueba = train_test_split(np.arange(len(df)), test_size=0.25, stratify=y_falsa, random_state=semilla)
        modelo = LogisticRegression(max_iter=1000).fit(X_sel[idx_ent], y_falsa.iloc[idx_ent])
        fuera.append(modelo.score(X_sel[idx_prueba], y_falsa.iloc[idx_prueba]))
        pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("sel", SelectKBest(chi2, k=50)), ("clf", LogisticRegression(max_iter=1000))])
        dentro.append(pipe.fit(df["texto"].iloc[idx_ent], y_falsa.iloc[idx_ent]).score(df["texto"].iloc[idx_prueba], y_falsa.iloc[idx_prueba]))
    return np.mean(fuera), np.mean(dentro)


filas = []
for idioma in ("spa", "ita"):
    fuera, dentro = fuga_seleccion(tickets[idioma])
    filas.append({"idioma": idioma, "selección fuera de la Pipeline": round(fuera, 3),
                  "selección dentro de la Pipeline": round(dentro, 3), "azar": 0.25})
    assert fuera > dentro + 0.05                       # la fuga infla el resultado de forma clara
print(pd.DataFrame(filas).to_string(index=False))

**¿Qué observas?** Con etiquetas **aleatorias** (sin nada que aprender), el modelo con la selección hecha fuera acierta 0,411 en español y 0,443 en italiano: **mucho más que el azar (0,25)**, y solo porque las 50 palabras se eligieron viendo también las etiquetas de la prueba. Dentro de la `Pipeline` el resultado vuelve a rondar el azar (0,253 y 0,258), que es lo correcto. **Regla práctica:** todo paso que aprende algo de los datos (vectorizador, selección de rasgos, escalado, reducción de dimensión) va **dentro** de la `Pipeline`.

---
## 5. Métricas: precisión, *recall*, F1 y matriz de confusión

Para cada clase, a partir de la **matriz de confusión** (filas = clase real, columnas = clase predicha):

- **Precisión** = aciertos de la clase / todo lo que el modelo etiquetó como esa clase (suma de la columna).
- **Recall** = aciertos de la clase / todos los que realmente son de esa clase (suma de la fila).
- **F1** = media armónica de precisión y recall: `2 · P · R / (P + R)`.
- **F1 macro** = media simple de los F1 por clase (todas pesan igual). **F1 ponderado** = media ponderada por el número de ejemplos reales de cada clase (el *soporte*).

Primero, a mano, con una matriz pequeña de 10 predicciones.

In [ ]:
CLASES_M = ["cuenta", "devoluciones", "envio"]
M = np.array([[4, 1, 0],          # real cuenta: 4 bien, 1 confundido con devoluciones
              [1, 2, 0],          # real devoluciones
              [0, 1, 1]])         # real envio
aciertos = np.diag(M)
precision = aciertos / M.sum(axis=0)
recall = aciertos / M.sum(axis=1)
f1 = 2 * precision * recall / (precision + recall)
soporte = M.sum(axis=1)
f1_macro_mano = f1.mean()
f1_pond_mano = (f1 * soporte).sum() / soporte.sum()
print(pd.DataFrame({"precisión": precision.round(4), "recall": recall.round(4), "F1": f1.round(4), "soporte": soporte}, index=CLASES_M))
print("F1 macro:", round(f1_macro_mano, 4), "| F1 ponderado:", round(f1_pond_mano, 4), "| exactitud:", aciertos.sum() / M.sum())

# Comprobación: reconstruimos las etiquetas a partir de la matriz y usamos scikit-learn
y_real = [CLASES_M[i] for i in range(3) for j in range(3) for _ in range(M[i, j])]
y_pred = [CLASES_M[j] for i in range(3) for j in range(3) for _ in range(M[i, j])]
p, r, f, s = precision_recall_fscore_support(y_real, y_pred, labels=CLASES_M)
assert np.allclose(p, precision) and np.allclose(r, recall) and np.allclose(f, f1) and (s == soporte).all()
assert np.isclose(f1_score(y_real, y_pred, average="macro"), f1_macro_mano)
assert np.isclose(f1_score(y_real, y_pred, average="weighted"), f1_pond_mano)
print("Coincide con scikit-learn ✔")

**¿Qué observas?** `envio` tiene precisión 1,0 (lo que marcó como `envio` era `envio`) pero recall 0,5 (solo encontró 1 de 2). Son cosas distintas: la precisión mira **lo que el modelo dice**, el recall **lo que existe**. El F1 macro (0,6794) y el ponderado (0,7048) difieren porque `cuenta` (F1 más alto) tiene más peso en el ponderado. Ahora la aplicamos a los tickets reales, con `classification_report` y la matriz de confusión del modelo de regresión logística.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
for eje, idioma in zip(ejes, ("spa", "ita")):
    y_prueba = DIV[idioma][3]
    pred = PREDICCIONES[("LogisticRegression", idioma)]
    print("=== ", idioma, " ===")
    print(classification_report(y_prueba, pred, digits=3, zero_division=0))
    matriz = confusion_matrix(y_prueba, pred, labels=ETIQUETAS)
    assert matriz.sum() == len(y_prueba) and (matriz.sum(axis=1) == 25).all()
    im = eje.imshow(matriz, cmap="Blues")
    eje.set_xticks(range(4)); eje.set_xticklabels(ETIQUETAS, rotation=30)
    eje.set_yticks(range(4)); eje.set_yticklabels(ETIQUETAS)
    for i in range(4):
        for j in range(4):
            eje.text(j, i, matriz[i, j], ha="center", va="center", color="white" if matriz[i, j] > 15 else "black")
    eje.set_xlabel("clase predicha"); eje.set_ylabel("clase real")
    eje.set_title(f"Matriz de confusión ({idioma})")
    fuera_diag = matriz.copy(); np.fill_diagonal(fuera_diag, 0)
    i, j = np.unravel_index(fuera_diag.argmax(), fuera_diag.shape)
    print("Confusión más frecuente:", ETIQUETAS[i], "→", ETIQUETAS[j], f"({fuera_diag[i, j]} tickets)")
plt.tight_layout()
plt.show()

**¿Qué observas?** Como la prueba está equilibrada (25 por clase), la exactitud, el F1 macro y el ponderado **coinciden** o casi. La diagonal de la matriz concentra la mayoría de los tickets; los errores se reparten y muestran qué categorías se confunden: en español, el fallo más frecuente es `devoluciones` → `facturacion` (4 tickets, porque un reembolso suena a dinero y a cobro); en italiano ningún par pasa de 2 tickets. Recuerda que el 10-15 % de los tickets contiene palabras típicas de otra categoría: parte de estos errores está en los datos, no en el modelo. **Cuando las clases no están equilibradas, esto cambia:** es lo siguiente.

---
## 6. Clases desbalanceadas: la exactitud engaña

Los tickets están equilibrados, así que **creamos el desbalance nosotros**, submuestreando cada categoría a un número distinto (100 de `facturacion`, 50 de `envio`, 20 de `cuenta` y 10 de `devoluciones`; no se crea ningún archivo nuevo). Es un experimento guiado: entrenamos con y sin `class_weight="balanced"` y miramos exactitud, F1 macro y ponderado y el *recall* de cada clase.

In [ ]:
CUOTAS = {"facturacion": 100, "envio": 50, "cuenta": 20, "devoluciones": 10}


def submuestrear(df, cuotas, semilla=42):
    """Se queda con cuotas[categoria] tickets de cada categoría."""
    partes = [g.sample(n=cuotas[c], random_state=semilla) for c, g in df.groupby("categoria")]
    return pd.concat(partes).reset_index(drop=True)


filas, informes = [], {}
for idioma in ("spa", "ita"):
    des = submuestrear(tickets[idioma], CUOTAS)
    assert des["categoria"].value_counts().to_dict() == CUOTAS
    X_ent, X_prueba, y_ent, y_prueba = train_test_split(des["texto"], des["categoria"], test_size=0.3, stratify=des["categoria"], random_state=42)
    trivial = DummyClassifier(strategy="most_frequent").fit(X_ent, y_ent).predict(X_prueba)
    for peso in (None, "balanced"):
        modelo = crear_pipeline(LogisticRegression(max_iter=1000, class_weight=peso)).fit(X_ent, y_ent)
        pred = modelo.predict(X_prueba)
        informes[(idioma, peso)] = classification_report(y_prueba, pred, digits=3, zero_division=0)
        _, recall_clase, _, _ = precision_recall_fscore_support(y_prueba, pred, labels=ETIQUETAS, zero_division=0)
        fila = {"idioma": idioma, "class_weight": str(peso), "exactitud": round(accuracy_score(y_prueba, pred), 3),
                "F1 macro": round(f1_score(y_prueba, pred, average="macro"), 3), "F1 ponderado": round(f1_score(y_prueba, pred, average="weighted"), 3)}
        fila.update({f"recall {c}": round(r, 2) for c, r in zip(ETIQUETAS, recall_clase)})
        filas.append(fila)
    print(idioma, "prueba:", dict(y_prueba.value_counts().sort_index()), "| exactitud siempre «facturacion»:", round(accuracy_score(y_prueba, trivial), 3))
print(pd.DataFrame(filas).to_string(index=False))
print()
print("Informe completo, español, sin pesos:")
print(informes[("spa", None)])

**¿Qué observas?** Sin pesos, la **exactitud parece aceptable** (0,704 en español, 0,667 en italiano; siempre decir «facturacion» ya daría 0,556) pero el **F1 macro se hunde** (0,371 y 0,335) y el *recall* de `cuenta` y `devoluciones` es **0**: el modelo no detecta nunca las clases pequeñas. La exactitud las oculta porque pesan poco. Con `class_weight="balanced"` el F1 macro sube a 0,796 y 0,776. **Cuidado:** la prueba solo tiene 6 tickets de `cuenta` y 3 de `devoluciones`, así que cada uno mueve mucho las cifras; lo confirmamos con validación cruzada en la sección 8. **Regla práctica:** con clases desbalanceadas mira el F1 macro y el recall por clase, no solo la exactitud.

---
## 7. Leer los coeficientes… con sentido crítico

En la regresión logística cada palabra tiene un **coeficiente por clase**: los positivos empujan hacia esa clase y los negativos en contra. Es una de sus grandes ventajas: se puede leer. Pero **leer no es creer**. Mira las palabras más pesadas de cada categoría, entrenando con todo el corpus solo para inspeccionar.

In [ ]:
def top_coeficientes(df, k=10):
    """Las k palabras con mayor coeficiente de cada clase (unigramas)."""
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))]).fit(df["texto"], df["categoria"])
    palabras = pipe.named_steps["tfidf"].get_feature_names_out()
    coef = pipe.named_steps["clf"].coef_
    return {clase: [(palabras[j], round(float(coef[i, j]), 2)) for j in np.argsort(coef[i])[::-1][:k]]
            for i, clase in enumerate(pipe.named_steps["clf"].classes_)}


TOP = {idioma: top_coeficientes(df) for idioma, df in tickets.items()}
for idioma in ("spa", "ita"):
    print("===", idioma, "===")
    for clase, lista in TOP[idioma].items():
        print(f"{clase:12s}", ", ".join(f"{p} ({v})" for p, v in lista))

**¿Qué observas?** La mayoría son palabras con sentido (`factura`, `paquete`, `devolución`, `contraseña`…). Pero fíjate en las que **no** son temáticas: palabras funcionales como `mi`, `quiero`, `sin`, `non`, `voglio`, `in`, `la`. Reflejan las **plantillas de redacción** de cada categoría (quien devuelve dice «quiero»), no su tema. Y hay algo más: los tickets tienen **artefactos**. Entre el 43 % y el 47 % termina con una frase muy corta, un «veredicto» (`Sin seguimiento.`, `Accesso bloccato.`), y cerca del 6 % con una etiqueta de asunto (`Asunto: cobro.`, `Oggetto: addebito.`). Lo comprobamos.

### Experimento guiado: ¿cuánto se apoya el modelo en el veredicto?

Con `separar_veredicto` separamos la última frase si tiene 4 palabras o menos. Primero contamos cuántos tickets la tienen y qué parte de las palabras más pesadas aparece en ese veredicto; después **quitamos el veredicto** y volvemos a medir con validación cruzada.

In [ ]:
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    partes = df["texto"].map(separar_veredicto)
    cuerpos = partes.map(lambda t: t[0])
    veredictos = partes.map(lambda t: t[1])
    con_veredicto = (veredictos != "").sum()
    con_asunto = df["texto"].str.contains(r"(?:Asunto|Oggetto):").sum()
    print(f"{idioma}: tickets con frase final corta = {con_veredicto} ({100 * con_veredicto / len(df):.1f} %), con etiqueta de asunto = {con_asunto} ({100 * con_asunto / len(df):.1f} %)")
    assert con_veredicto > 100 and con_asunto > 10

    # Para las 8 palabras más pesadas de cada clase: ¿en cuántos tickets aparecen y en cuántos están en el veredicto?
    filas = []
    for clase, lista in TOP[idioma].items():
        for palabra, coef in lista[:8]:
            patron = r"\b" + re.escape(palabra) + r"\b"
            en_texto = df["texto"].str.lower().str.contains(patron)
            en_veredicto = veredictos.str.lower().str.contains(patron)
            filas.append({"clase": clase, "palabra": palabra, "coef.": coef, "tickets con la palabra": int(en_texto.sum()),
                          "en el veredicto": int(en_veredicto.sum())})
    tabla = pd.DataFrame(filas)
    tabla["% en veredicto"] = (100 * tabla.iloc[:, 4] / tabla.iloc[:, 3]).round(0)
    print(tabla.sort_values("% en veredicto", ascending=False).head(8).to_string(index=False))

    # Ablación: validación cruzada con y sin veredicto
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc_con = cross_val_score(crear_pipeline(LogisticRegression(max_iter=1000)), df["texto"], df["categoria"], cv=skf).mean()
    acc_sin = cross_val_score(crear_pipeline(LogisticRegression(max_iter=1000)), cuerpos, df["categoria"], cv=skf).mean()
    print(f"   exactitud (CV 5 particiones) con el veredicto: {acc_con:.4f} | sin el veredicto: {acc_sin:.4f} | diferencia: {acc_con - acc_sin:.4f}")
    assert acc_con > acc_sin

**¿Qué observas?** Varias de las palabras más pesadas aparecen sobre todo en esas frases finales, y al **quitar el veredicto la exactitud baja** (0,8825 → 0,8475 en español; 0,9050 → 0,8550 en italiano). El modelo se apoya en un atajo que un ticket real casi nunca ofrece: nadie termina su mensaje con `Tema de garantía.`. **Lección:** un coeficiente alto dice qué usa el modelo, **no** qué es lo importante en el mundo real. Antes de interpretar una lista de palabras pesadas, busca artefactos (etiquetas, firmas, plantillas, fechas) y compara el modelo con y sin ellos. Y recuerda: como los tickets en español y en italiano son adaptaciones cercanas, este hallazgo es **una sola observación vista dos veces**, no una réplica independiente.

---
## 8. Validación cruzada estratificada

Una sola partición entrenamiento/prueba da una cifra que depende de la suerte del corte. La **validación cruzada** de `k` particiones divide los datos en `k` trozos y usa cada uno como prueba una vez; la **estratificada** (`StratifiedKFold`) mantiene en cada trozo la proporción de clases. Pasar la `Pipeline` entera a `cross_val_score` hace que el vectorizador se ajuste **dentro** de cada partición. Primero, por qué estratificar y barajar: sobre el conjunto desbalanceado de la sección 6, que está ordenado por categoría.

In [ ]:
des_spa = submuestrear(tickets["spa"], CUOTAS)            # ordenado por categoría
for nombre, particionador in (("KFold sin barajar", KFold(n_splits=5)),
                              ("StratifiedKFold", StratifiedKFold(n_splits=5, shuffle=True, random_state=42))):
    print(nombre)
    for k, (_, idx_prueba) in enumerate(particionador.split(des_spa, des_spa["categoria"]), 1):
        print("  partición", k, dict(des_spa.iloc[idx_prueba]["categoria"].value_counts().sort_index()))
skf_test = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for _, idx_prueba in skf_test.split(des_spa, des_spa["categoria"]):
    assert des_spa.iloc[idx_prueba]["categoria"].value_counts()["devoluciones"] == 2      # 10 devoluciones repartidas: 2 por partición

**¿Qué observas?** Sin barajar, las particiones de prueba tienen composiciones muy distintas: la 2 contiene **solo `envio`** y la 4 y la 5 **solo `facturacion`**, y la 1 no tiene ningún `facturacion`; el modelo se evalúa con clases que casi no ha visto al entrenar. Con `StratifiedKFold` cada partición conserva las proporciones (unos 20 `facturacion`, 10 `envio`, 4 `cuenta`, 2 `devoluciones`). Ahora comparamos los tres modelos con 5 particiones estratificadas, en los tickets equilibrados, y el efecto de `class_weight` en los desbalanceados.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
filas = []
for nombre, fabrica in MODELOS.items():
    fila = {"modelo": nombre}
    for idioma in ("spa", "ita"):
        puntos = cross_val_score(crear_pipeline(fabrica()), tickets[idioma]["texto"], tickets[idioma]["categoria"], cv=skf)
        fila[f"{idioma} media"] = round(puntos.mean(), 3)
        fila[f"{idioma} desv. típica"] = round(puntos.std(), 3)
        assert len(puntos) == 5 and puntos.mean() > 0.7
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

print()
print("Conjunto desbalanceado (español), F1 macro con CV estratificada de 5 particiones:")
for peso in (None, "balanced"):
    puntos = cross_val_score(crear_pipeline(LogisticRegression(max_iter=1000, class_weight=peso)), des_spa["texto"], des_spa["categoria"], cv=skf, scoring="f1_macro")
    print(f"  class_weight={peso}: media {puntos.mean():.3f} ± {puntos.std():.3f}")

**¿Qué observas?** Con validación cruzada, los tres modelos quedan en 0,865-0,905 y **la desviación típica entre particiones (0,021-0,044) es del mismo orden que las diferencias entre modelos**: no hay un ganador claro. Es la cifra que conviene dar (media ± desviación) y se acerca a la línea base ≈ 0,88-0,89 del curso. En el conjunto desbalanceado, `class_weight="balanced"` mejora el F1 macro (0,333 → 0,673), lo que confirma con más datos de prueba lo que sugería la partición única de la sección 6.

## 9. Resumen y siguiente paso

- **Naive Bayes multinomial** = prior × producto de `P(palabra | clase)`; el **suavizado de Laplace** (`alpha = 1`) suma un pseudorrecuento para evitar probabilidades cero. Se calculó a mano y coincide con `MultinomialNB`.
- **Regresión logística** y **`LinearSVC`** sobre TF-IDF rinden parecido en estos corpus; con una sola partición no se distingue al mejor.
- La **`Pipeline`** ajusta todo solo con el entrenamiento. Con el TF-IDF la fuga fue casi nula, pero con un paso que usa etiquetas (selección de rasgos) fue enorme: todo lo que aprende va dentro.
- **Precisión, recall, F1** (macro y ponderado) y **matriz de confusión** dicen dónde falla el modelo. Con clases desbalanceadas la **exactitud engaña**: mira el F1 macro y el recall por clase.
- Los **coeficientes** se leen con sentido crítico: los artefactos (veredictos, etiquetas de asunto) se cuelan entre las palabras más pesadas.
- **`StratifiedKFold`** da una estimación más estable y respeta las proporciones de clase.
- Recuerda: corpus pequeños, sintéticos y con español/italiano casi paralelos: las cifras son un *ejemplo ilustrativo*.

**Siguiente:** en la clase 6, *Análisis de sentimiento clásico*, aplicarás estos clasificadores a reseñas con negación e ironía y los compararás con un léxico. Antes, haz el notebook de práctica de esta clase.